In [3]:
import numpy as np
import matplotlib.pyplot as plt
import diffrax as dfx
import equinox as eqx
import jax.numpy as jnp
import optimistix as optx
import scipy.constants as const

In [ ]:
def vector_field(t, y, parameters):
    I, freq, sigma_g, sigma_s, sigma_t, t10, t13, t21, t30, t43 = parameters
    ns0, ns1, ns2, nt1, nt2 = y
    dns0 = -(sigma_g * I * ns0) / (const.h * freq) + (ns1 / t10) + (nt1 / t30)
    dns1 = (sigma_g * I * ns0) / (const.h * freq) - (ns1 / t10) - (ns1 / t13) + (ns2 / t21) - (sigma_s * I * ns1) / (const.h * freq)
    dns2 = (sigma_s * I * ns1) / (const.h * freq) - (ns2 / t21)
    dnt1 = -(sigma_t * I * nt1) / (const.h * freq) + (nt2 / t43) + (ns1 / t13) - (nt1 / t30)
    dnt2 = (sigma_t * I * nt1) / (const.h * freq) - (nt2 / t43)
    return jnp.array([dns0, dns1, dns2, dnt1, dnt2])

def solve(
        parameters,
        y0,
        dt,
        saveat
):
    term = dfx.ODETerm(vector_field)
    solver = dfx.Kvaerno5()
    stepsize_controller = dfx.PIDController(rtol=1e-6, atol=1e-12)
    sol = dfx.diffeqsolve(
        term,
        solver,
        t0=0,
        t1=dt,
        dt0=1e-10,
        y0=y0,
        args=(parameters,),
        saveat=saveat,
        stepsize_controller=stepsize_controller
    )
    return sol.ys